# Savant 05 self-play sampling

This notebook writes training conversations in the model's own words. The base model answers each development table several times through a vLLM server, and a run is kept only when it is grounded, names every pair the guarded engine confirmed and names no false pair. Failed code steps and rejected answers stay in the context but carry no training labels.

With PROBE set to True the run is small, so it measures speed and the keep rate before the full run spends real GPU quota. Development seeds start at 4,000,000, far from the bench seed 20261105.


In [ ]:
import glob
import json
import os
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

build = sorted(glob.glob("/kaggle/input/**/BUILD", recursive=True))
assert build and Path(build[0]).read_text().strip() == "EXPECTED_BUILD", "stale bench dataset, stop"
wheel = sorted(glob.glob("/kaggle/input/**/savant-*.whl", recursive=True))[-1]
t0 = time.time()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", wheel, "vllm"], check=True)
install_seconds = round(time.time() - t0)
print("installed in", install_seconds, "s")
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout)

In [ ]:
# Settings for this run.
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
PROBE = False  # a small run that measures speed before the full run
TABLES = 24 if PROBE else 500
SAMPLES = 4  # answers per table
SEED = 4_000_000  # development seeds, the generator refuses the bench range
TEMPERATURE = 0.7
WORKERS = 32  # conversations in flight across both servers
KEEP_MESSAGES = 6
PER_TABLE = 2  # at most this many kept runs per table
MAX_NEW_TOKENS = 768
FULL_RUNS = 500 * SAMPLES  # used to estimate the cost of the full run

In [ ]:
def start(gpu, port, extra=()):
    log = open(f"/kaggle/working/vllm-{gpu}.log", "w")  # noqa: SIM115, the server keeps it
    cmd = [
        "vllm", "serve", MODEL_ID,
        "--served-model-name", "savant",
        "--dtype", "half",
        "--max-model-len", "8192",
        "--gpu-memory-utilization", "0.90",
        "--port", str(port),
        *extra,
    ]  # fmt: skip
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}
    return subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env)


def healthy(port, procs, limit=900):
    end = time.time() + limit
    while time.time() < end:
        if any(p.poll() is not None for p in procs):
            return False
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{port}/health", timeout=5)
            return True
        except Exception:
            time.sleep(10)
    return False


PORTS = [8000, 8001]
t0 = time.time()
mode = "default"
procs = [start(g, p) for g, p in enumerate(PORTS)]
if not all(healthy(p, procs) for p in PORTS):
    # CUDA graph capture is the usual failure on older cards, so retry without it.
    for p in procs:
        p.kill()
    time.sleep(10)
    mode = "enforce-eager"
    procs = [start(g, p, ["--enforce-eager"]) for g, p in enumerate(PORTS)]
    ok = all(healthy(p, procs) for p in PORTS)
    if not ok:
        for g in range(len(PORTS)):
            print(f"---- vllm-{g}.log")
            print(Path(f"/kaggle/working/vllm-{g}.log").read_text()[-4000:])
        Path("/kaggle/working/selfplay.json").write_text(
            json.dumps({"vllm": "failed to start", "install_seconds": install_seconds})
        )
        raise SystemExit("vLLM did not start on these GPUs")
start_seconds = round(time.time() - t0)
print("vLLM ready in", start_seconds, "s, mode", mode)

In [ ]:
import itertools
import threading

from savant.llm import OpenAICompatible
from savant.selfplay import collect, dev_tasks, select, summarize

ports = itertools.cycle(PORTS)
lock = threading.Lock()


def client():
    with lock:
        port = next(ports)
    return OpenAICompatible(
        f"http://127.0.0.1:{port}/v1", "savant", temperature=TEMPERATURE, max_tokens=MAX_NEW_TOKENS
    )


log = Path("/kaggle/working/runs.jsonl")
log.write_text("")
done = []


def on_record(r):
    row = {k: v for k, v in r.items() if k not in ("messages", "learn")}
    row["transcript"] = r["messages"]
    with lock:
        done.append(r)
        with log.open("a") as f:
            f.write(json.dumps(row, default=str) + "\n")
        if len(done) % 20 == 0:
            kept = sum(x["keep"] for x in done)
            print(f"{len(done)} runs, {kept} kept, {time.time() - t1:.0f}s", flush=True)


t0 = time.time()
tasks = dev_tasks(TABLES, SEED)
t1 = time.time()
records = collect(
    tasks,
    client,
    samples=SAMPLES,
    workers=WORKERS,
    keep_messages=KEEP_MESSAGES,
    scan_workers=os.cpu_count() or 2,
    on_record=on_record,
)
seconds = time.time() - t0
chosen = select(records, PER_TABLE)
with open("/kaggle/working/trajectories.jsonl", "w") as f:
    for r in chosen:
        keys = ("task", "kind", "sample", "question", "turns", "failed", "found", "confirmed")
        f.write(
            json.dumps({**{k: r[k] for k in keys}, "messages": r["messages"], "learn": r["learn"]})
            + "\n"
        )
summary = summarize(records, chosen)
summary.update(
    {
        "probe": PROBE,
        "vllm_mode": mode,
        "install_seconds": install_seconds,
        "start_seconds": start_seconds,
        "sampling_seconds": round(seconds),
        "runs_per_hour": round(len(records) / seconds * 3600),
        "estimated_hours_for_full_run": round(FULL_RUNS / len(records) * seconds / 3600, 2),
        "estimated_trajectories_for_full_run": round(len(chosen) / TABLES * 500),
    }
)
Path("/kaggle/working/selfplay.json").write_text(json.dumps(summary, indent=1))
print(json.dumps(summary, indent=1))
for p in procs:
    p.kill()